# Convolutional Neural Networks and Image Classification

Our first adventure with nontabular data.

In [ ]:
import torch
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib.ticker as mtick
import torch.nn as nn
from torch.nn import Conv2d, MaxPool2d, Parameter
from torch.nn.functional import relu
from torchvision import models
import torch.optim as optim
from torchsummary import summary
import torch.nn as nn
from torch.nn import ReLU

plt.style.use('seaborn-v0_8-whitegrid')

Our data for today is the [Sign Language MNIST](https://www.kaggle.com/datasets/datamunge/sign-language-mnist) data set, which I retrieved from Kaggle. This data set poses a challenge: can we train a model to recognize a letter of American Sign Language from a hand gesture?

In [ ]:
train_url = "https://raw.githubusercontent.com/PhilChodrow/ml-notes/main/data/sign-language-mnist/sign_mnist_train.csv"
test_url = "https://raw.githubusercontent.com/PhilChodrow/ml-notes/main/data/sign-language-mnist/sign_mnist_test.csv"

df_train = pd.read_csv(train_url)
df_val   = pd.read_csv(test_url)

Natively, this data set comes to us as a data frame in which each column represents a pixel. Each image has 28x28 pixels, so there are 784 pixel columns, plus one column for the label (the letter being signed). Let’s take a look at the data frame:

In [ ]:
df_train.head()

In principle, we’re already able to perform machine learning tasks on data in this format – treat each column as a feature and we’re ready to go. However, this format makes it very difficult to see *relationships* between these features. Importantly, a key feature of image data is that *nearby* pixels are often related to each other in important ways. There’s no hope to capture that idea from this data frame, because we can’t even tell *which* pixels are nearby each other. Let’s therefore reshape the data into something more like its native pixel format.

In [ ]:
def prep_data(df):
    n, p = df.shape[0], df.shape[1] - 1
    y = torch.tensor(df["label"].values)
    X = df.drop(["label"], axis = 1)
    X = torch.tensor(X.values)
    X = torch.reshape(X, (n, 1, 28, 28))
    X = X / 255

    return X, y

X_train, y_train = prep_data(df_train)
X_val, y_val     = prep_data(df_val)

In [ ]:
print("Training data shapes:")
print(X_train.shape, y_train.shape)
print("Validation data shapes:")
print(X_val.shape, y_val.shape)

We’ve shaped the data into a 4-dimensional tensor, with dimensions <span class="column-margin margin-aside">The *channel* refers to the number of color channels in the image. Colors represented in **R**ed, **G**reen, and **B**lue color format (RGB) have 3 channels, one for each of R, G, and B. Greyscale images like the ones we’re working with today have only one channel.</span>

$$
\begin{aligned}
    (\text{image index}, \text{channel}, \text{width}, \text{height})\;.
\end{aligned}
$$

So, to interpret `X_train.shape`, we have 27,455 images in the training data, each with 1 channel (grayscale), and each image is 28 pixels wide and 28 pixels high.

> **Beyond data frames**
>
> So far in this class, we’ve almost exclusively considered data arranged in 2-dimensional structures like matrices or data frames with dimensions $n\times d$, where $n$ is the number of data observations and $d$ is the number of features. As we move into deep learning, we’re going to see more examples of data sets like this one in which the 2-dimensional representation of data is inappropriate. We’ll therefore need to expand our thinking to more general tensors with more than two dimensions, like the one we have here.

Figure 1: Example images from the sign-language MNIST data set, with their corresponding class labels.

In [ ]:
ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def show_images(X, y, rows, cols, channel = 0):

    fig, axarr = plt.subplots(rows, cols, figsize = (2*cols, 2*rows))
    for i, ax in enumerate(axarr.ravel()):
        ax.imshow(X[i, channel].detach().cpu(), cmap = "Greys_r")
        ax.set(title = f"{ALPHABET[y[i]]}")
        ax.axis("off")
    plt.tight_layout()

show_images(X_train, y_train, 5, 5)

Let’s take a look at the distribution of characters in the training data:

Figure 2: Distribution of class labels in the training data. There are no “J”s or “Z”s in this data because these characters require motion rather than a static gesture.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize = (6, 2))
letters, counts = torch.unique(y_train, return_counts = True)
proportions = counts / counts.sum()
proportions

ax.scatter(letters, proportions, edgecolor = "steelblue")
ax.set_xticks(range(26))
ax.set_xticklabels(list(ALPHABET))
ax.set(xlabel = "Letter", ylabel = "Frequency")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals = 1))
ax.set_ylim(0, 0.06)

The most frequent letter (“R”) in this data comprises no more than 5% of the entire data set. So, as a minimal aim, we would like a model that gets the right character at least 5% of the time.

## Data Prep

### Sidebar: GPU Acceleration in Torch

As we’ve seen from the last several lectures, deep learning models involve a **lot** of linear algebra in order to compute predictions and gradients. This means that deep models, even more than many other machine learning models, strongly benefit from hardware that is good at doing linear algebra *fast*. As it happens, graphics processing units (GPUs) are very, very good at fast linear algebra. <span class="column-margin margin-aside">The reason that GPUs are so good at this is that they were originally optimized for rendering complex graphics in e.g. animation and video games, and this involves lots of linear algebra.</span> So, it’s very helpful when running our models to have access to GPUs; using a GPU can often result in up to 10x speedups. While some folks can use GPUs on their personal laptops, another common option for learning purposes is to use a cloud-hosted GPU. My personal recommendation is Google Colab, and I’ll supply links that allow you to open lecture notes in Colab and use their GPU runtimes.

The following torch code checks whether there is a GPU available to PyTorch, and if so, sets a variable called `device` to log this fact. We’ll make sure that both our data and our models fully live on the same device when doing model training.

In [ ]:
# TODO

Now that we’ve configured the device, we are going to need to place both our data and our models on the device.

In [ ]:
# TODO

### Data Loaders

As we saw when studying [modern approaches to optimization loops](31-gradient-methods.qmd), it’s convenient to loop through our data in batches and perform an optimization step after each batch. The following code defines a function that creates a data loader for our training and validation data sets.

In [ ]:
def make_data_loader(X, y, batch_size = 32): 
    return torch.utils.data.DataLoader(
                torch.utils.data.TensorDataset(X, y),
                batch_size = 32,
                shuffle = True
    )

train_loader = make_data_loader(X_train, y_train)
val_loader   = make_data_loader(X_val, y_val)

As a reminder, it’s possible to loop through elements of the data loader using syntax like:

In [ ]:
for X_batch, y_batch in train_loader:
    # do something with X_batch 
    # and y_batch

### Logistic Regression Baseline

Let’s go ahead and train a logistic regression model on this data. This model will make *no use* of the spatial structure of the pixels. Relative to our previous logistic regression implementations, the main difference here is that we need to structure the model to accept input data whose single instance has dimensions (1, 28, 28) rather than (784,). We can achieve this by including a `Flatten` layer in our model, which will take the 1x28x28 pixel input and flatten it into a 784-dimensional vector before applying the linear transformation.

We can make our code a bit more concise by enclosing each of our layers inside an `nn.Sequential` container, which allows us to treat the entire sequence of layers as a single layer which we then call in the `forward` method. <span class="column-margin margin-aside">Relative to previous implementations of logistic regression, you might notice that this implementation does not contain a call to the logistic sigmoid. The reason for this is that we are going to use `torch`’s built-in `nn.CrossEntropyLoss()` as our loss function for training. For reasons of numerical stability, this function is structured to apply the cross-entropy loss, so we don’t need to do it in the model itself.</span>

In [ ]:
# TODO

### Model Inspection

When constructing nontrivial deep learning models, it can be helpful to inspect them in order to get a sense for their structure and complexity level (especially the number of parameters). <span class="column-margin margin-aside">For example, could you read off from the above model definition how many parameters are included in the model?</span> There are multiple ways to approach this, including several utilities which visualize the computational graph. Here, we’ll use the `torchsummary` package, which provides a nice summary of the model’s layers and parameters and includes a parameter count. To call the `summary` function, we need to specify the expected dimension of a single piece of data (including the channel):

In [ ]:
# TODO

Even though we are looking at a simple logistic regression model, our parameter count is already in the tens of thousands!

The code block below allows us to evaluate a model’s performance by computing its accuracy and a confusion matrix on a specified data loader. There’s also a helper function for plotting the confusion matrix.

In [ ]:
def evaluate(model, data_loader, multichannel = False, print_message = True): 
    ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    
    # confusion matrix
    conf_mat = torch.zeros((26, 26), dtype = torch.int32)

    loss_fn = nn.CrossEntropyLoss()

    with torch.no_grad(): 
        for X_batch, y_batch in data_loader:
            scores = model(X_batch)

            loss = loss_fn(scores, y_batch)

            y_pred = torch.argmax(scores, dim = 1)

            for i in range(len(y_batch)):
                conf_mat[y_batch[i], y_pred[i]] += 1

    acc = torch.diag(conf_mat).sum() / conf_mat.sum()
    if print_message:
        print(f"Accuracy: {acc:.2%}")

    return acc, loss, conf_mat

In [ ]:
def plot_confusion_mat(conf_mat, ax, title = "Confusion Matrix"):
    im = ax.imshow(conf_mat.cpu(), cmap = "Blues", origin = "upper")
    # Show all ticks and label them with the respective list entries
    ax.set_xticks(torch.arange(len(ALPHABET)))
    ax.set_yticks(torch.arange(len(ALPHABET)))
    ax.set_xticklabels(list(ALPHABET))
    ax.set_yticklabels(list(ALPHABET))
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")

    # Rotate the tick labels and set their alignment.
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
             rotation_mode="anchor")

    # Loop over data dimensions and create text annotations.
    for i in range(conf_mat.shape[0]):
        for j in range(conf_mat.shape[1]):
            text = ax.text(j, i, conf_mat[i, j].item(),
                           ha="center", va="center", color="black", size = 6)

    ax.set_title(title)
    ax.grid(False)

Figure 3: Confusion matrix for the untrained logistic regression model.

In [ ]:
fig, ax = plt.subplots(figsize = (7, 7))
acc, loss, conf_mat = evaluate(model, val_loader, print_message = False)
plot_confusion_mat(conf_mat, ax, title = f"Confusion Matrix (acc = {acc:.2%})")

Obviously this model does not classify the data very impressively at all, since it hasn’t been trained yet. Let’s implement a training loop. This loop will perform model updates and track the model’s accuracy over time. <span class="column-margin margin-aside">It would also be reasonable to track the average loss per batch over time. This would require an extra call to `model.forward` on the validation set, which we’re not going to do here because it increases the training time substantially.</span>

In [ ]:
def train(model, k_epochs = 1, print_every = 2000, **opt_kwargs):

    # loss function is cross-entropy (multiclass logistic)
    loss_fn = nn.CrossEntropyLoss()

    # optimizer is SGD with momentum
    optimizer = optim.SGD(model.parameters(), **opt_kwargs)

    # initialize list of accuracies to 
    train_accuracy = []
    val_accuracy   = []
    train_loss     = []
    val_loss       = []

    for epoch in range(k_epochs):

        for i, data in enumerate(train_loader):
            X, y = data

            # clear any accumulated gradients
            optimizer.zero_grad()

            # compute the loss
            y_pred = model(X)
            loss   = loss_fn(y_pred, y)

            # compute gradients and carry out an optimization step
            loss.backward()
            optimizer.step()
            
        train_acc, train_l, train_cm = evaluate(model, data_loader = train_loader)
        val_acc, val_l, val_cm     = evaluate(model, data_loader = val_loader)
        
        train_accuracy += [train_acc]
        val_accuracy   += [val_acc]
        train_loss     += [train_l]
        val_loss       += [val_l]

    return train_accuracy, train_loss, val_accuracy, val_loss

Now let’s train the model and visualize its performance over time:

In [ ]:
# TODO

Figure 4: Training curves for the logistic regression model. The left panel shows the training and validation loss over time, and the right panel shows the training and validation accuracy over time.

In [ ]:
fig, axarr = plt.subplots(1, 2, figsize = (8, 3.5))

ax = axarr[0]
ax.plot(train_loss, color = "black", label = "Training")
ax.plot(val_loss, color = "firebrick", label = "Validation")
ax.set_title("Cross-Entropy Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")

ax = axarr[1]
ax.plot(train_accuracy, color = "black", label = "Training")
ax.plot(val_accuracy, color = "firebrick", label = "Validation")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Classification Accuracy")
ax.set(ylim = (0, 1))
plt.tight_layout()
l = ax.legend()

This model is not yet done training, and additional epochs might improve the performance. For the purposes of these notes we won’t push it further than that.

Already, the confusion matrix for the trained model looks much better:

Figure 5: Confusion matrix for the logistic regression model.

In [ ]:
acc, loss, conf_mat = evaluate(model, val_loader, print_message = False)
fig, ax = plt.subplots(figsize = (7, 7))
plot_confusion_mat(conf_mat, ax, title = f"Confusion Matrix (acc = {acc:.2%})")

Although we can see that the model is *often* successful, 54% accuracy leaves a lot of room for improvement. Can we do better?

## Convolutional Neural Networks

A common approach to feature extraction in images is to apply a *convolutional kernel*. <span class="column-margin margin-aside">Convolutional kernels are not related in any way to positive-definite kernels used in kernel classifiers, another important ML topic.</span> A convolutional kernel is a component of a vectorization pipeline which is specifically suited to the structure of images. In particular, *images are fundamentally spatial*. We might want to construct data features which reflect not just the value of an individual pixel, but also the values of pixels nearby that one.

The idea of an image convolution is pretty simple. We define a square *kernel matrix* containing some numbers, and we “slide it over” the input data. At each location, we multiply the data values by the kernel matrix values, and add them together. Here’s an illustrative diagram:

### Kernel Convolutions

<figure id="fig-kernel-convolution">
<img src="https://d2l.ai/_images/correlation.svg" />
<figcaption>Figure 6: Image from <a href="https://d2l.ai/chapter_convolutional-neural-networks/conv-layer.html"><em>Dive Into Deep Learning</em></a>.</figcaption>
</figure>

In this example, the value of 19 is computed as $0\times 0 + 1\times 1 + 3\times 2 + 4\times 3 = 19$.

### Manual Kernel Convolutions for Feature Extraction

For a long time, a common approach to image classification and related computer vision tasks was to hand-engineer a set of convolutional kernels designed to extract certain specific features of interest from the image. For example, here are some kernels designed to extract vertical, horizontal, and diagonal features from an image:

In [ ]:
vertical = torch.tensor([0, 0, 5, 0, 0]).repeat(5, 1) - 1.0
diag1    = torch.eye(5)*5 - 1
horizontal = torch.transpose(vertical, 1, 0)
diag2    = diag1.flip(1)

fig, ax = plt.subplots(1, 4)
for i, kernel in enumerate([vertical, horizontal, diag1, diag2]):
    ax[i].imshow(kernel, vmin = -1.5, vmax = 2)
    ax[i].axis("off")
    ax[i].set(title = f'{["Vertical", "Horizontal", "Diagonal Down", "Diagonal Up"][i]}')

When we apply these convolutional kernels to an image, we obtain a new image in which the value of each pixel corresponds to the “alignment” of the image with the kernel at that point. Here are some examples:

Figure 7: Example of four kernel convolutions applied to five sample input images.

In [ ]:
def apply_convolutions(X): 

    # this is actually a neural network layer -- we'll learn how to use these
    # in that context soon 
    conv1 = Conv2d(1, 4, 5) # 1 input channel, 4 output channels, 5x5 kernels

    # need to disable gradients for this layer
    for p in conv1.parameters():
        p.requires_grad = False

    # replace kernels in layer with our custom ones
    conv1.weight[0, 0] = Parameter(vertical)
    conv1.weight[1, 0] = Parameter(horizontal)
    conv1.weight[2, 0] = Parameter(diag1)
    conv1.weight[3, 0] = Parameter(diag2)

    # apply to input data and disable gradients
    return conv1(X).detach()

def kernel_viz(pipeline):

    fig, ax = plt.subplots(5, 5, figsize = (8, 8))

    X_convd = pipeline(X_train)

    for i in range(5): 
        for j in range(5):
            if i == 0: 
                ax[i,j].imshow(X_train[j, 0])
            
            else: 
                ax[i, j].imshow(X_convd[j,i-1])
            
            ax[i,j].tick_params(
                        axis='both',      
                        which='both',     
                        bottom=False,     
                        left=False,
                        right=False,         
                        labelbottom=False, 
                        labelleft=False)
            ax[i,j].grid(False)
            ax[i, 0].set(ylabel = ["Original", "Vertical", "Horizontal", "Diag Down", "Diag Up"][i])

kernel_viz(apply_convolutions)

In principle, these convolutions could be used to define “scores” for each image: for example, summing up the “horizontal” values could give an image a score reflecting the prevalence of horizontal lines in the image.

A limitation of this approach is that we have to engineer all our kernels in advance. Wouldn’t it be simpler if we could simply initialize the kernels randomly and let the data tell us what the useful kernels might be?

### Learnable Kernels

Fortunately, neural networks give us a framework for doing exactly that. The key insight is that the kernel convolution operation is, fundamentally, just a sequence of pairwise multiplications followed by an addition. This means that the convolution is a *linear operation*. This means:

**Kernel Convolution is a Matrix Multiplication**

Actually writing down the matrix multiplication formula for kernel convolution is complex and involves “doubly-index matrices,” so we won’t do that here. The key point is that, since the convolution operation is a matrix multiplication, we can treat it with the same framework as we have been using for other linear models – we just need to throw it in as a layer in a neural network.

### A Minimal Convolutional Model

Just inserting a convolutional layer on its own won’t lead to substantial gains because, as we saw, composing linear layers doesn’t actually add that much. Instead, we can compose the convolutional layer with a nonlinearity (e.g. ReLU) and a final linear layer to get a minimal convolutional model. Since our output layer is a `Linear` layer, we still need to `Flatten` the output of the convolutional layer before feeding it into the linear layer.

In [ ]:
# TODO

Let’s instantiate a model and take a look.

In [ ]:
model = SmallConvNet().to(device)
summary(model, input_size=(1, 28, 28))

How does this model perform?

In [ ]:
k_epochs = 1

train_accuracy, train_loss, val_accuracy, val_loss = train(model, k_epochs = k_epochs,  lr = 0.01,  momentum = 0.9)

Figure 8: Training curves for the minimal convolutional neural network.

In [ ]:
fig, axarr = plt.subplots(1, 2, figsize = (8, 3.5))

ax = axarr[0]
ax.plot(train_loss, color = "black", label = "Training")
ax.plot(val_loss, color = "firebrick", label = "Validation")
ax.set_title("Cross-Entropy Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")

ax = axarr[1]
ax.plot(train_accuracy, color = "black", label = "Training")
ax.plot(val_accuracy, color = "firebrick", label = "Validation")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Classification Accuracy")
ax.set(ylim = (0, None))
plt.tight_layout()
l = ax.legend()

Figure 9: Confusion matrix for the minimal convolutional neural network. .

In [ ]:
acc, loss, conf_mat = evaluate(model, val_loader, print_message = False)
fig, ax = plt.subplots(figsize = (7, 7))
plot_confusion_mat(conf_mat, ax, title = f"Confusion Matrix (acc = {acc:.2%})")

The minimal model is already able to somewhat exceed the performance of the logistic regression model, even after fewer epochs of training.

##### ISSUE IS AFTER HERE

### More Complex Convolutional Models

A common approach to building more complex convolutional models is to stack multiple convolutional layers on top of each other, with nonlinearities in between. This allows the model to learn more complex features at different levels of abstraction. For example, the first convolutional layer might learn to detect edges, while the second convolutional layer might learn to detect combinations of edges that form shapes, and the third convolutional layer might learn to detect combinations of shapes that form objects.

#### Pooling

An issue with the stacking approach, however, is that the data remains very large throughout the pipeline, with each convolution reducing the data just by a few pixels in each dimension. This also prevents convolutional kernels at later layers from combining data from far away regions in the image. To address this, let’s *reduce* the data in a nonlinear way. We’ll do this with max pooling. You can think of it as a kind of “summarization” step in which we intentionally make the current output somewhat “blockier.” Technically, it involves sliding a window over the current batch of data and picking only the largest element within that window. Here’s an example of how this looks:

<figure id="fig-max-pooling">
<img src="https://computersciencewiki.org/images/8/8a/MaxpoolSample2.png" />
<figcaption>Figure 10: Illustration of max-pooling. <em>Image credit: Computer Science Wiki</em></figcaption>
</figure>

A useful effect of pooling is that it reduces the number of features in our data. In the image above, we reduce the number of features by a factor of $2\times 2 = 4$.

Let’s now construct a complex model that layers convolutional layers, nonlinearities, and pooling layers. <span class="column-margin margin-aside">There’s nothing special about the engineering of this model: I just stacked a few convolution and pooling layers together and then added a few more linear outputs.</span>

In [ ]:
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.pipeline = torch.nn.Sequential(
            nn.Conv2d(1, 100, 5),
            nn.MaxPool2d(2, 2),
            ReLU(),
            nn.Conv2d(100, 50, 3),
            nn.MaxPool2d(2, 2),
            ReLU(),
            nn.Conv2d(50, 50, 3),
            nn.MaxPool2d(2, 2),
            ReLU(),
            nn.Flatten(),
            nn.Linear(50, len(ALPHABET))
        )

    def forward(self, x):
        return self.pipeline(x)

This model has considerable additional complexity, indicated by its parameter count.

In [ ]:
model = ConvNet().to(device)
summary(model, input_size=(1, 28, 28))

Let’s try training the model and seeing how we do.

### BEFORE HERE

In [ ]:
train_accuracy, train_loss, val_accuracy, val_loss = train(model, k_epochs = k_epochs,  lr = 0.01,  momentum = 0.9)

Figure 11: Training curves for the convolutional neural network.

In [ ]:
fig, axarr = plt.subplots(1, 2, figsize = (8, 3.5))

ax = axarr[0]
ax.plot(train_loss, color = "black", label = "Training")
ax.plot(val_loss, color = "firebrick", label = "Validation")
ax.set_title("Cross-Entropy Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")

ax = axarr[1]
ax.plot(train_accuracy, color = "black", label = "Training")
ax.plot(val_accuracy, color = "firebrick", label = "Validation")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Classification Accuracy")
ax.set(ylim = (0, None))
plt.tight_layout()
l = ax.legend()

### SPLIT

Figure 12: Confusion matrix for the convolutional neural network.

In [ ]:
acc, loss, conf_mat = evaluate(model, val_loader, print_message = False)
fig, ax = plt.subplots(figsize = (7, 7))
plot_confusion_mat(conf_mat, ax, title = f"Confusion Matrix (acc = {acc:.2%})")

The additional complexity of this model enables it to achieve much higher accuracy than the previous models we’ve discussed, although more thorough training runs would be necessary for a full assessment.

## Inspecting Learned Features

Like we saw [last time](40-intro-deep.qmd), it’s possible to inspect the features learned by a neural network at different levels of abstraction. Let’s see some of the features learned by the model for a single image:

Figure 13: Example image used as base for the experiment in <a href="#fig-learned-features" class="quarto-xref">Figure 14</a>.

In [ ]:
X_orig, y_orig = next(iter(train_loader))
fig, ax = plt.subplots(1, 1)
ax.imshow(X_orig[0,0].detach().cpu(), cmap = "Greys")
ax.axis("off")

In the code block below, we show the outputs of different layers of the model when applied to this original image. Each layer’s output can be thought of as a different “representation” of the original image, with different features extracted at each layer.

Figure 14: Sample learned features at varying levels of abstraction from the convolutional model, using <a href="#fig-original-for-features" class="quarto-xref">Figure 13</a> as a base. Note that the features in each row are not necessarily directly related to each other.

In [ ]:
def feature_at_layer(model, X_sample, layer_num):

    for i, layer in enumerate(model.pipeline): 
        X_sample = layer(X_sample)
        if i == layer_num: 
            break
    
    return X_sample

fig, axarr = plt.subplots(3, 4, figsize = (10, 8))

for i in range(3): 

    for ix, j in enumerate([0, 1, 3, 4]): 
        X_sample = X_orig.clone()
        X_sample = feature_at_layer(model, X_sample, j)
        axarr[i, ix].imshow(X_sample[0][i].detach().cpu())
        axarr[i, ix].axis("off")
        layer_name = model.pipeline[j].__class__.__name__
        axarr[0, ix].set_title(f"Layer {j}: {layer_name}")

Interpreting these learned features can be tricky and is not generally recommended without context and many additional experiments.

### Onward

In the next lecture, we’ll consider some additional practical considerations that arise when working with spatially-structured data and convolutional neural networks.